# Sorani (Kurdish) Voice Training - Piper TTS (Seçenek A)

Bu not defteri, kadın Sorani (ckb) sesini Piper TTS'e eğitir. Üç veri kaynağı:
- `commonvoice` — Common Voice ckb kadın konuşmacı (önerilen, kotasız)
- `synthetic` — kurdishtts ile üretilmiş sentetik Sorani (`synthetic.zip`)
- `recording` — Recording Studio kayıtları (wav+txt zip)

Eğitim sonunda `model.onnx` + `model.onnx.json` üretilir; bilgisayarda `deploy/stt/models/`'e koyup TTS'i devreye alırsın.

Adımlar: 1) kurulum  2) veri (HF onayı + indir)  3) hazırla  4) EĞİT  5) ONNX + indir

In [ ]:
# 0) GPU kontrol
!nvidia-smi --query-gpu=name --format=csv,noheader 2>/dev/null || echo "GPU yok"

In [ ]:
!pip install -q piper-training datasets
print("kuruldu")

## 2) Veri kaynağını seç

In [ ]:
# 2) VERİ KAYNAĞI
# "commonvoice" | "synthetic" | "recording"
DATA_SOURCE = "commonvoice"
TARGET_HOURS = 1.0
print("Kaynak:", DATA_SOURCE)

### ⚠️ HuggingFace erişimini ayarla (Common Voice için)

Common Voice HuggingFace'te **onay (gated)** isteyebilir. Çalıştırmadan önce:

1. [huggingface.co](https://huggingface.co) → ücretsiz hesap aç + giriş.
2. Şu sayfada **Agree and access repository** ile kabul et:
   `https://huggingface.co/datasets/mozilla-foundation/common_voice_17_0`
3. **Settings → Access Tokens** → `Read` yetkili `hf_...` token oluştur.
4. Aşağıdaki hücreye token'ı yapıştır.

Onay anında olmazsa biraz bekle (manuel onay istenebilir). `synthetic` / `recording` kullanıyorsan bu adımı geç.

In [ ]:
# 2.5) HuggingFace giriş (sadece commonvoice için)
HF_TOKEN = ""   # ← token'ını yapıştır)
if HF_TOKEN:
    from huggingface_hub import login
    login(HF_TOKEN, add_to_git_credential=False)
    print("HF giriş yapıldı")

### (A) Common Voice — otomatik kadın konuşmacı
`DATA_SOURCE == "commonvoice"` ise bu hücreyi çalıştır.

In [ ]:
# 2.A) Common Voice ckb indir (kadın konuşmacı)
import os, glob, librosa, soundfile as sf
import numpy as np
from datasets import load_dataset

os.makedirs("data", exist_ok=True)
auth = {"token": HF_TOKEN} if HF_TOKEN else {}
ds = None
for ver in ["common_voice_17_0", "common_voice_13_0", "common_voice_12_0"]:
    try:
        print("deneniyor:", ver)
        ds = load_dataset("mozilla-foundation/" + ver, "ckb", split="train", **auth)
        break
    except Exception as e:
        print("hata:", e)
if ds is None:
    print("Common Voice erişilemedi. HF 'Agree' yaptıysan token doğru; yoksa (B) Sentetik yolunu kullan.")
    raise SystemExit("erisim yok")

df = ds.to_pandas()
f = df[df["gender"] == "female"]
print("kadın kayıt:", len(f))
if len(f) == 0:
    print("Kadın kayıt yok — 'any' ile deneniyor")
    f = df
top = f["client_id"].value_counts().idxmax()
sub = f[f["client_id"] == top]
print("seçilen konuşmacı:", top, "kayıt:", len(sub))
target = int(TARGET_HOURS * 3600 / 3)
sub = sub.head(target)
for i, row in enumerate(sub.itertuples()):
    arr = row.audio["array"]; sr = row.audio["sampling_rate"]
    a = librosa.resample(arr.astype("float32"), orig_sr=sr, target_sr=22050)
    if a.ndim > 1: a = a.mean(axis=1)
    p = f"data/cv_{i:05d}.wav"
    sf.write(p, a, 22050)
    open(p.replace(".wav", ".txt"), "w", encoding="utf-8").write(row.sentence)
print("hazır:", len(glob.glob("data/*.wav")))

### (B) Sentetik / Recording — zip yükle
`DATA_SOURCE == synthetic` veya `recording` ise bu hücreyi çalıştır (`synthetic.zip` yükle).

In [ ]:
# 2.B) Sentetik/Recording zip yükle
from google.colab import files
import glob, os, zipfile
os.makedirs("data", exist_ok=True)
up = files.upload()
for fn in up:
    if fn.endswith(".zip"):
        import zipfile
        with zipfile.ZipFile(fn, "r") as z:
            z.extractall("data")
        print("açıldı:", fn)
print("data/wav:", len(glob.glob("data/*.wav")))

## 3) Piper verisini hazırla + EĞİT

In [ ]:
# 3) Piper metadata hazırla
!piper.train_prepare --dataset-dir data --sample-rate 22050 --language ckb --config-out config.json
print("config ok")

In [ ]:
# 4) EĞİT (Colab GPU ~1-3 saat)
!piper.train --config-dir . --run-name sorani-female
print("eğitim bitti")

In [ ]:
# 5) ONNX + indir
!piper.export_onnx --run-dir runs --output-dir output --dataset-dir data
!ls -la output/
from google.colab import files
import glob
for f in sorted(glob.glob("output/*")):
    print("indir:", f)
    files.download(f)